# Task 2 – Bottleneck & Corridor Audit
## Graph-Based Network Intelligence for Delhivery Logistics

Performs a full bottleneck and corridor audit of the logistics network using NetworkX.

### Deliverables
1. **Weighted Directed Graph** — edge weights = median actual/OSRM delay ratio per corridor
2. **In-Degree & Out-Degree** — separate directed analysis; hub role classification
3. **Betweenness Centrality** — structural chokepoint hubs
4. **Clustering Coefficients** — undirected + directed; identifies structural bridges
5. **Chronic Delay Corridors** — actual time exceeds OSRM by >20% (delay ratio > 1.2)
6. **Stratified Delay Analysis** — route type × time-of-day breakdown
7. **SLA Breach Contribution** — corridor-level and hub-level ranking
8. **Composite Bottleneck Ranking** — 40% betweenness + 40% SLA breach + 20% degree
9. **Visualizations** — network subgraph, corridor bar charts, hub ranking, degree distribution

In [ ]:
import pandas as pd
import networkx as nx
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import os
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
print('All libraries imported successfully.')

## 1. Data Loading

In [ ]:
df = pd.read_csv('../data/processed/delivery_data_features.csv')
print('Dataset loaded:', df.shape[0], 'rows x', df.shape[1], 'columns')

required_cols = [
    'source_center', 'destination_center',
    'segment_actual_time', 'segment_osrm_time',
    'route_type', 'trip_hour'
]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError('Missing required columns: ' + str(missing))
print('All required columns present.')
print()
print('Route type distribution:')
print(df['route_type'].value_counts())

## 2. Delay Ratio Computation & SLA Breach Definition

- **Delay ratio** = `segment_actual_time / segment_osrm_time`
- A shipment is an **SLA breach** when `delay_ratio > 1.2` (actual exceeds OSRM by >20%)
- Rows where `segment_osrm_time <= 0` are excluded to avoid division by zero

In [ ]:
df_valid = df[df['segment_osrm_time'] > 0].copy()
print('Valid rows (segment_osrm_time > 0):', len(df_valid), 'of', len(df))

df_valid['delay_ratio'] = df_valid['segment_actual_time'] / df_valid['segment_osrm_time']
df_valid['is_sla_breach'] = df_valid['delay_ratio'] > 1.2

df_valid['time_period'] = pd.cut(
    df_valid['trip_hour'],
    bins=[0, 6, 12, 18, 24],
    labels=['Night', 'Morning', 'Afternoon', 'Evening'],
    right=False
)

breach_rate = round(df_valid['is_sla_breach'].mean() * 100, 1)
breach_count = int(df_valid['is_sla_breach'].sum())
print()
print('Overall SLA breach rate (actual > OSRM by >20%):', breach_rate, '%')
print('Total SLA breaches:', breach_count)
print()
print('Delay ratio statistics:')
print(df_valid['delay_ratio'].describe().round(3))

## 3. Corridor-Level Aggregation

Aggregate per corridor (source → destination pair):
- **median_delay_ratio**: primary edge weight for the graph
- **sla_breach_count**: number of shipments where actual > OSRM by >20%
- **is_chronic**: True if `median_delay_ratio > 1.2` (corridor is systematically late)

In [ ]:
corridor_stats = df_valid.groupby(
    ['source_center', 'destination_center']
).agg(
    median_delay_ratio=('delay_ratio', 'median'),
    mean_delay_ratio=('delay_ratio', 'mean'),
    median_actual_time=('segment_actual_time', 'median'),
    median_osrm_time=('segment_osrm_time', 'median'),
    shipment_count=('delay_ratio', 'count'),
    sla_breach_count=('is_sla_breach', 'sum'),
).reset_index()

corridor_stats['sla_breach_pct'] = (
    corridor_stats['sla_breach_count'] / corridor_stats['shipment_count'] * 100
).round(2)

corridor_stats['is_chronic'] = corridor_stats['median_delay_ratio'] > 1.2
corridor_stats['delay_above_osrm_pct'] = (
    (corridor_stats['median_delay_ratio'] - 1) * 100
).round(1)

corridor_stats['corridor'] = (
    corridor_stats['source_center'] + '->' + corridor_stats['destination_center']
)

total_sla_breaches = int(corridor_stats['sla_breach_count'].sum())
corridor_stats['sla_breach_contribution_pct'] = (
    corridor_stats['sla_breach_count'] / total_sla_breaches * 100
).round(4)

chronic_count = int(corridor_stats['is_chronic'].sum())
print('Corridor aggregation complete.')
print('Total corridors:', len(corridor_stats))
print('Chronic delay corridors (median delay ratio > 1.2):', chronic_count)
print('Pct chronically delayed:', round(chronic_count / len(corridor_stats) * 100, 1), '%')
print('Total SLA breaches across network:', total_sla_breaches)

## 4. Build Weighted Directed Graph

Facilities = nodes, corridors = directed edges.  
Edge weight = **median actual/OSRM delay ratio** (higher = more delayed).  
Additional edge attributes: `shipment_count`, `sla_breach_count`, `sla_breach_pct`, `is_chronic`.

In [ ]:
G = nx.DiGraph()

for _, row in corridor_stats.iterrows():
    G.add_edge(
        row['source_center'],
        row['destination_center'],
        weight=float(row['median_delay_ratio']),
        median_actual_time=float(row['median_actual_time']),
        median_osrm_time=float(row['median_osrm_time']),
        shipment_count=int(row['shipment_count']),
        sla_breach_count=int(row['sla_breach_count']),
        sla_breach_pct=float(row['sla_breach_pct']),
        sla_breach_contribution_pct=float(row['sla_breach_contribution_pct']),
        is_chronic=bool(row['is_chronic']),
        delay_above_osrm_pct=float(row['delay_above_osrm_pct'])
    )

chronic_edges = sum(1 for u, v, d in G.edges(data=True) if d['is_chronic'])
total_shipments_graph = sum(d['shipment_count'] for u, v, d in G.edges(data=True))
total_sla_graph = sum(d['sla_breach_count'] for u, v, d in G.edges(data=True))
out_degrees = [d for _, d in G.out_degree()]

print('Weighted Directed Graph constructed:')
print('  Nodes (Hubs):          ', G.number_of_nodes())
print('  Edges (Corridors):     ', G.number_of_edges())
print('  Chronic delay edges:   ', chronic_edges)
print('  Is directed:           ', G.is_directed())
print('  Avg out-degree:        ', round(np.mean(out_degrees), 2))
print('  Max out-degree:        ', max(out_degrees))
print('  Network density:       ', round(nx.density(G), 6))

## 5. Network Summary

In [ ]:
network_summary = pd.DataFrame({
    'Metric': [
        'Total Hubs (Nodes)',
        'Total Corridors (Edges)',
        'Chronic Delay Corridors (Actual > OSRM by >20%)',
        'Pct Corridors Chronically Delayed',
        'Total Shipments Analyzed',
        'Total SLA Breaches',
        'Overall SLA Breach Rate Pct',
        'Avg Network Delay Ratio',
        'Max Corridor Delay Ratio',
        'Network Density',
    ],
    'Value': [
        G.number_of_nodes(),
        G.number_of_edges(),
        chronic_edges,
        str(round(chronic_edges / G.number_of_edges() * 100, 1)) + '%',
        total_shipments_graph,
        int(total_sla_graph),
        str(round(total_sla_graph / total_shipments_graph * 100, 1)) + '%',
        round(corridor_stats['median_delay_ratio'].mean(), 3),
        round(corridor_stats['median_delay_ratio'].max(), 3),
        round(nx.density(G), 6),
    ]
})

network_summary.to_csv('../reports/network_summary.csv', index=False)
print('Network summary saved.')
network_summary

## 6. In-Degree & Out-Degree Analysis

- **In-degree**: number of corridors arriving at a hub (receiving hub)
- **Out-degree**: number of corridors departing from a hub (dispatching hub)
- **Weighted degree**: sum of delay ratios on adjacent edges (operational load proxy)
- **Hub role**: classified by ratio of in vs out connections

In [ ]:
in_deg = dict(G.in_degree())
out_deg = dict(G.out_degree())
in_deg_c = nx.in_degree_centrality(G)
out_deg_c = nx.out_degree_centrality(G)

in_deg_weighted = {
    n: sum(G[u][n]['weight'] for u in G.predecessors(n))
    for n in G.nodes()
}
out_deg_weighted = {
    n: sum(G[n][v]['weight'] for v in G.successors(n))
    for n in G.nodes()
}

degree_df = pd.DataFrame({
    'hub': list(G.nodes()),
    'in_degree': [in_deg[n] for n in G.nodes()],
    'out_degree': [out_deg[n] for n in G.nodes()],
    'total_degree': [in_deg[n] + out_deg[n] for n in G.nodes()],
    'in_degree_centrality': [round(in_deg_c[n], 6) for n in G.nodes()],
    'out_degree_centrality': [round(out_deg_c[n], 6) for n in G.nodes()],
    'weighted_in_degree': [round(in_deg_weighted[n], 4) for n in G.nodes()],
    'weighted_out_degree': [round(out_deg_weighted[n], 4) for n in G.nodes()],
})

def classify_hub(row):
    if row['out_degree'] > row['in_degree'] * 1.5:
        return 'Distribution Hub'
    elif row['in_degree'] > row['out_degree'] * 1.5:
        return 'Aggregation Hub'
    elif row['total_degree'] >= 20:
        return 'Major Transit Hub'
    return 'Transit Hub'

degree_df['hub_role'] = degree_df.apply(classify_hub, axis=1)
degree_df = degree_df.sort_values('total_degree', ascending=False).reset_index(drop=True)

top_in_degree = (
    degree_df.sort_values('in_degree', ascending=False)
    [['hub', 'in_degree', 'in_degree_centrality', 'weighted_in_degree', 'hub_role']]
    .head(30).reset_index(drop=True)
)
top_out_degree = (
    degree_df.sort_values('out_degree', ascending=False)
    [['hub', 'out_degree', 'out_degree_centrality', 'weighted_out_degree', 'hub_role']]
    .head(30).reset_index(drop=True)
)

degree_df.head(30).to_csv('../reports/degree_centrality_hubs.csv', index=False)
top_in_degree.to_csv('../reports/in_degree_hubs.csv', index=False)
top_out_degree.to_csv('../reports/out_degree_hubs.csv', index=False)
print('Degree reports saved.')
print()
print('Top 10 Hubs by Total Degree:')
print(degree_df[['hub', 'in_degree', 'out_degree', 'total_degree', 'hub_role']].head(10).to_string(index=False))
print()
print('Hub role distribution:')
print(degree_df['hub_role'].value_counts())

## 7. Betweenness Centrality

Measures how often a hub sits on the shortest path between other hubs.  
High betweenness = structural chokepoint: removing or upgrading this hub affects the most routes.  
Uses k=300 sampled approximation for efficiency on large graphs.

In [ ]:
print('Computing betweenness centrality (k=300 sample approximation)...')
betweenness = nx.betweenness_centrality(G, k=300, seed=42, normalized=True)

betweenness_df = (
    pd.DataFrame(betweenness.items(), columns=['hub', 'betweenness_centrality'])
    .sort_values('betweenness_centrality', ascending=False)
    .reset_index(drop=True)
)

betweenness_df.head(30).to_csv('../reports/betweenness_centrality_hubs.csv', index=False)
print('Betweenness centrality saved.')
print()
print('Top 10 Hubs by Betweenness Centrality:')
print(betweenness_df.head(10).to_string(index=False))

## 8. Clustering Coefficients

- **High clustering coefficient**: hub is in a tightly-connected cluster (redundant paths exist)
- **Low clustering + high betweenness**: hub is a structural bridge — the most dangerous bottleneck type  
  (single point of failure with no alternative routes)
- Computed both on the undirected version (standard) and the directed graph

In [ ]:
print('Computing clustering coefficients...')
G_undirected = G.to_undirected()
clust_undirected = nx.clustering(G_undirected)
clust_directed = nx.clustering(G)

clustering_df = pd.DataFrame({
    'hub': list(G.nodes()),
    'clustering_coefficient': [round(clust_undirected[n], 6) for n in G.nodes()],
    'directed_clustering_coefficient': [round(clust_directed[n], 6) for n in G.nodes()],
}).sort_values('clustering_coefficient', ascending=False).reset_index(drop=True)

avg_clust = nx.average_clustering(G_undirected)
transitivity = nx.transitivity(G)

clustering_df.head(30).to_csv('../reports/clustering_coefficients.csv', index=False)
print('Clustering coefficients saved.')
print('Network average clustering coefficient:', round(avg_clust, 4))
print('Network transitivity (directed):        ', round(transitivity, 4))
print()
print('Top 10 Hubs by Clustering Coefficient (densely clustered):')
print(clustering_df.head(10).to_string(index=False))
print()
structural_bridges = clustering_df[
    clustering_df['clustering_coefficient'] < 0.05
].sort_values('clustering_coefficient').head(10)
print('Bottom 10 Hubs by Clustering (potential structural bridges):')
print(structural_bridges.to_string(index=False))

## 9. Chronic Delay Corridor Audit (Actual > OSRM by >20%)

A corridor is **chronically delayed** if its median delay ratio > 1.2 (actual time at least 20% above OSRM estimate).  
Ranked by median delay ratio descending. Also shows SLA breach counts per corridor.

In [ ]:
chronic_corridors = corridor_stats[corridor_stats['is_chronic']].copy()
chronic_corridors = chronic_corridors.sort_values('median_delay_ratio', ascending=False)

chronic_display = chronic_corridors[[
    'corridor', 'source_center', 'destination_center',
    'median_delay_ratio', 'delay_above_osrm_pct',
    'median_actual_time', 'median_osrm_time',
    'shipment_count', 'sla_breach_count', 'sla_breach_pct',
    'sla_breach_contribution_pct'
]].reset_index(drop=True)

chronic_display.to_csv('../reports/bottleneck_corridors.csv', index=False)
print('Chronic delay corridors saved.')
print('Total chronic corridors:', len(chronic_display))
print('Pct of all corridors:   ', round(len(chronic_display) / len(corridor_stats) * 100, 1), '%')
print()
print('Top 20 Chronic Delay Corridors (Actual time exceeds OSRM by >20%):')
chronic_display.head(20)

## 10. Stratified Delay Analysis: Route Type x Time of Day

Edge weights in the project requirement are stratified by route type AND time of day.  
This cell computes median delay ratios and SLA breach rates for each combination.

In [ ]:
stratified = df_valid.groupby(
    ['route_type', 'time_period'],
    observed=True
).agg(
    median_delay_ratio=('delay_ratio', 'median'),
    mean_delay_ratio=('delay_ratio', 'mean'),
    shipment_count=('delay_ratio', 'count'),
    sla_breach_count=('is_sla_breach', 'sum'),
    sla_breach_rate_pct=('is_sla_breach', lambda x: round(x.mean() * 100, 2)),
).round(3).reset_index()
stratified = stratified.sort_values(
    ['route_type', 'median_delay_ratio'], ascending=[True, False]
)

stratified.to_csv('../reports/stratified_delay_analysis.csv', index=False)
print('Stratified delay analysis saved.')
print()
print('Delay Ratio by Route Type x Time of Day:')
print(stratified.to_string(index=False))

## 11. SLA Breach Contribution by Corridor

Ranks corridors by their **absolute contribution to total network SLA breaches**.  
`sla_breach_contribution_pct` = (corridor SLA breaches / total network SLA breaches) x 100.  
`cumulative_breach_pct` shows cumulative % — useful for Pareto analysis.

In [ ]:
sla_corridors = corridor_stats[[
    'corridor', 'source_center', 'destination_center',
    'shipment_count', 'sla_breach_count', 'sla_breach_pct',
    'sla_breach_contribution_pct', 'median_delay_ratio', 'is_chronic'
]].sort_values('sla_breach_count', ascending=False).reset_index(drop=True)

sla_corridors['cumulative_breach_pct'] = (
    sla_corridors['sla_breach_contribution_pct'].cumsum().round(3)
)

sla_corridors.head(50).to_csv('../reports/sla_breach_corridors.csv', index=False)
print('SLA breach corridors saved.')
print()
print('Top 20 Corridors by SLA Breach Contribution (% of all network breaches):')
print(sla_corridors.head(20).to_string(index=False))

## 12. SLA Breach Contribution by Hub

- **out_sla_breach_count**: shipments dispatched FROM this hub that breached SLA (hub's dispatch responsibility)
- **in_sla_breach_count**: shipments arriving AT this hub already late (upstream delay indicator)
- **sla_breach_contribution_pct**: out-breach count as % of total network breaches

In [ ]:
hub_breach_data = {}
for node in G.nodes():
    out_breaches = sum(G[node][v]['sla_breach_count'] for v in G.successors(node))
    out_shipments = sum(G[node][v]['shipment_count'] for v in G.successors(node))
    in_breaches = sum(G[u][node]['sla_breach_count'] for u in G.predecessors(node))
    in_shipments = sum(G[u][node]['shipment_count'] for u in G.predecessors(node))
    hub_breach_data[node] = {
        'out_sla_breach_count': out_breaches,
        'out_shipment_count': out_shipments,
        'in_sla_breach_count': in_breaches,
        'in_shipment_count': in_shipments,
    }

hub_sla_df = (
    pd.DataFrame.from_dict(hub_breach_data, orient='index')
    .reset_index().rename(columns={'index': 'hub'})
)

hub_sla_df['out_breach_rate_pct'] = (
    hub_sla_df['out_sla_breach_count'] /
    hub_sla_df['out_shipment_count'].replace(0, np.nan) * 100
).fillna(0).round(2)

hub_sla_df['in_breach_rate_pct'] = (
    hub_sla_df['in_sla_breach_count'] /
    hub_sla_df['in_shipment_count'].replace(0, np.nan) * 100
).fillna(0).round(2)

hub_sla_df['sla_breach_contribution_pct'] = (
    hub_sla_df['out_sla_breach_count'] / total_sla_breaches * 100
).round(4)

hub_sla_df = hub_sla_df.sort_values(
    'out_sla_breach_count', ascending=False
).reset_index(drop=True)

hub_sla_df.head(50).to_csv('../reports/sla_breach_hubs.csv', index=False)
print('Hub SLA breach contribution saved.')
print()
print('Top 20 Hubs by SLA Breach Contribution (Outgoing Shipments):')
print(hub_sla_df.head(20).to_string(index=False))

## 13. Composite Bottleneck Hub Ranking

Combines three normalized scores:
- **40% Betweenness Centrality** — structural chokepoint importance
- **40% SLA Breach Count** — operational damage to delivery promises
- **20% Total Degree** — network connectivity exposure

This ranking is used to prioritize which hubs to upgrade first.

In [ ]:
def min_max_scale(series):
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series(0.5, index=series.index)
    return (series - mn) / (mx - mn)

btw_part = betweenness_df[['hub', 'betweenness_centrality']].copy()
sla_hub_part = hub_sla_df[[
    'hub', 'out_sla_breach_count', 'sla_breach_contribution_pct', 'out_breach_rate_pct'
]].copy()
deg_part = degree_df[[
    'hub', 'in_degree', 'out_degree', 'total_degree',
    'in_degree_centrality', 'out_degree_centrality', 'hub_role'
]].copy()
clust_part = clustering_df[['hub', 'clustering_coefficient']].copy()

bottleneck_ranking = (
    btw_part
    .merge(sla_hub_part, on='hub', how='left')
    .merge(deg_part, on='hub', how='left')
    .merge(clust_part, on='hub', how='left')
    .fillna(0)
)

bottleneck_ranking['norm_betweenness'] = min_max_scale(bottleneck_ranking['betweenness_centrality'])
bottleneck_ranking['norm_sla_breach'] = min_max_scale(bottleneck_ranking['out_sla_breach_count'])
bottleneck_ranking['norm_total_degree'] = min_max_scale(bottleneck_ranking['total_degree'])

bottleneck_ranking['composite_bottleneck_score'] = (
    0.40 * bottleneck_ranking['norm_betweenness'] +
    0.40 * bottleneck_ranking['norm_sla_breach'] +
    0.20 * bottleneck_ranking['norm_total_degree']
).round(4)

bottleneck_ranking = bottleneck_ranking.sort_values(
    'composite_bottleneck_score', ascending=False
).reset_index(drop=True)
bottleneck_ranking.index += 1

bottleneck_ranking.head(50).reset_index().rename(columns={'index': 'rank'}).to_csv(
    '../reports/hub_bottleneck_ranking.csv', index=False
)
print('Hub bottleneck ranking saved.')
print()
print('Top 20 Bottleneck Hubs (40% Betweenness + 40% SLA Breach + 20% Degree):')
display_cols = [
    'hub', 'composite_bottleneck_score', 'betweenness_centrality',
    'sla_breach_contribution_pct', 'total_degree', 'hub_role'
]
print(bottleneck_ranking[display_cols].head(20).to_string())

## 14. Visualizations

Four plots saved to `../reports/visualizations/`:
1. **Bottleneck Network Subgraph** — top 30 hubs, node size = composite score, edge color = delay ratio
2. **Chronic Corridors** — delay ratio bar + SLA breach contribution bar
3. **Hub Bottleneck Ranking** — composite score, betweenness, SLA breach side by side
4. **Degree Distribution** — in-degree and out-degree histograms

In [ ]:
os.makedirs('../reports/visualizations', exist_ok=True)
print('Visualizations directory ready.')

In [ ]:
# Visualization 1: Bottleneck Network Subgraph
top_n = 30
top_hubs = set(bottleneck_ranking.head(top_n)['hub'].tolist())
subgraph_nodes = set(top_hubs)
for hub in top_hubs:
    for nbr in list(G.predecessors(hub))[:2]:
        subgraph_nodes.add(nbr)
    for nbr in list(G.successors(hub))[:2]:
        subgraph_nodes.add(nbr)

H = G.subgraph(subgraph_nodes).copy()

node_scores = bottleneck_ranking.set_index('hub')['composite_bottleneck_score'].to_dict()
node_btw = bottleneck_ranking.set_index('hub')['betweenness_centrality'].to_dict()

nodes_list = list(H.nodes())
node_sizes = [200 + node_scores.get(n, 0.0) * 2500 for n in nodes_list]
node_color_vals = [node_btw.get(n, 0.0) for n in nodes_list]

edge_tuples = list(H.edges(data=True))
edge_weights = [d['weight'] for u, v, d in edge_tuples]
edge_widths = [0.5 + np.log1p(d['shipment_count']) * 0.25 for u, v, d in edge_tuples]

fig, ax = plt.subplots(figsize=(18, 14))
fig.patch.set_facecolor('#0f1117')
ax.set_facecolor('#0f1117')

pos = nx.spring_layout(H, k=2.5, seed=42, iterations=80)

edge_cmap = plt.cm.RdYlGn_r
ew_min = min(edge_weights) if edge_weights else 1.0
ew_max = max(edge_weights) if edge_weights else 2.0
edge_norm = mcolors.Normalize(vmin=ew_min, vmax=ew_max)
edge_colors_mapped = [edge_cmap(edge_norm(w)) for w in edge_weights]

nx.draw_networkx_edges(
    H, pos, ax=ax,
    edge_color=edge_colors_mapped,
    width=edge_widths,
    alpha=0.55,
    arrows=True,
    arrowsize=12
)

node_cmap = plt.cm.plasma
nc_min = min(node_color_vals) if node_color_vals else 0.0
nc_max = max(node_color_vals) if node_color_vals else 1.0
node_norm = mcolors.Normalize(vmin=nc_min, vmax=nc_max)
node_colors_mapped = [node_cmap(node_norm(v)) for v in node_color_vals]

nx.draw_networkx_nodes(
    H, pos, ax=ax,
    node_size=node_sizes,
    node_color=node_colors_mapped,
    alpha=0.9,
    linewidths=1.5,
    edgecolors='white'
)

label_hubs = set(bottleneck_ranking.head(15)['hub'].tolist())
labels = {n: n for n in H.nodes() if n in label_hubs}
nx.draw_networkx_labels(
    H, pos, labels=labels, ax=ax,
    font_size=6.5, font_color='white', font_weight='bold'
)

sm_edge = plt.cm.ScalarMappable(cmap=edge_cmap, norm=edge_norm)
sm_edge.set_array([])
cb1 = fig.colorbar(sm_edge, ax=ax, shrink=0.35, pad=0.01)
cb1.set_label('Corridor Delay Ratio', color='white', fontsize=9)
plt.setp(cb1.ax.yaxis.get_ticklabels(), color='white')

sm_node = plt.cm.ScalarMappable(cmap=node_cmap, norm=node_norm)
sm_node.set_array([])
cb2 = fig.colorbar(sm_node, ax=ax, shrink=0.35, pad=0.06)
cb2.set_label('Betweenness Centrality', color='white', fontsize=9)
plt.setp(cb2.ax.yaxis.get_ticklabels(), color='white')

ax.set_title(
    'Delhivery Network - Top Bottleneck Hub Subgraph  '
    '(Node size = composite score | Edge color = delay ratio)',
    color='white', fontsize=12, pad=15
)
ax.axis('off')
plt.tight_layout()
plt.savefig(
    '../reports/visualizations/bottleneck_network.png',
    dpi=150, bbox_inches='tight', facecolor='#0f1117'
)
plt.close()
print('Bottleneck network visualization saved.')

In [ ]:
# Visualization 2: Chronic Corridors + SLA Breach Contribution
top_chronic_viz = chronic_display.head(20).copy()
top_chronic_viz['short'] = top_chronic_viz['corridor'].str[:22] + '..'

sla_top20 = sla_corridors.head(20).copy()
sla_top20['short'] = sla_top20['corridor'].str[:22] + '..'

fig, axes = plt.subplots(1, 2, figsize=(18, 9))
fig.patch.set_facecolor('#0f1117')

ax1 = axes[0]
ax1.set_facecolor('#1a1d27')
colors1 = plt.cm.RdYlGn_r(np.linspace(0.2, 0.9, len(top_chronic_viz)))
ax1.barh(top_chronic_viz['short'][::-1].values, top_chronic_viz['median_delay_ratio'][::-1].values, color=colors1)
ax1.axvline(x=1.2, color='yellow', linestyle='--', linewidth=2, label='>20% SLA threshold (1.2x)')
ax1.set_xlabel('Median Delay Ratio (Actual / OSRM)', color='white', fontsize=10)
ax1.set_title('Top 20 Chronic Delay Corridors', color='white', fontsize=12)
ax1.tick_params(colors='white', labelsize=7)
for sp in ['top', 'right']:
    ax1.spines[sp].set_visible(False)
for sp in ['bottom', 'left']:
    ax1.spines[sp].set_color('#555')
ax1.legend(facecolor='#1a1d27', edgecolor='#555', labelcolor='white', fontsize=8)

ax2 = axes[1]
ax2.set_facecolor('#1a1d27')
ax2.barh(sla_top20['short'][::-1].values, sla_top20['sla_breach_contribution_pct'][::-1].values, color='#e07b39')
ax2.set_xlabel('% of Total Network SLA Breaches', color='white', fontsize=10)
ax2.set_title('Top 20 Corridors by SLA Breach Contribution', color='white', fontsize=12)
ax2.tick_params(colors='white', labelsize=7)
for sp in ['top', 'right']:
    ax2.spines[sp].set_visible(False)
for sp in ['bottom', 'left']:
    ax2.spines[sp].set_color('#555')

fig.suptitle('Corridor Audit - Chronic Delays and SLA Breach Contribution', color='white', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(
    '../reports/visualizations/chronic_corridors.png',
    dpi=150, bbox_inches='tight', facecolor='#0f1117'
)
plt.close()
print('Chronic corridors visualization saved.')

In [ ]:
# Visualization 3: Hub Bottleneck Ranking (3 panels)
top20 = bottleneck_ranking.head(20).reset_index(drop=True)
hub_labels = top20['hub'].tolist()
y_pos = np.arange(len(hub_labels))

fig, axes = plt.subplots(1, 3, figsize=(22, 10))
fig.patch.set_facecolor('#0f1117')

for ax in axes:
    ax.set_facecolor('#1a1d27')
    for sp in ['top', 'right']:
        ax.spines[sp].set_visible(False)
    for sp in ['bottom', 'left']:
        ax.spines[sp].set_color('#555')

axes[0].barh(y_pos, top20['composite_bottleneck_score'][::-1].values, color='#7c5cbf')
axes[0].set_yticks(y_pos)
axes[0].set_yticklabels(hub_labels[::-1], fontsize=7.5, color='white')
axes[0].set_xlabel('Composite Bottleneck Score', color='white', fontsize=9)
axes[0].set_title('Overall Bottleneck Score', color='white', fontsize=11)
axes[0].tick_params(colors='white')

axes[1].barh(y_pos, top20['betweenness_centrality'][::-1].values, color='#3a9bd5')
axes[1].set_yticks(y_pos)
axes[1].set_yticklabels(hub_labels[::-1], fontsize=7.5, color='white')
axes[1].set_xlabel('Betweenness Centrality', color='white', fontsize=9)
axes[1].set_title('Structural Importance (Betweenness)', color='white', fontsize=11)
axes[1].tick_params(colors='white')

axes[2].barh(y_pos, top20['sla_breach_contribution_pct'][::-1].values, color='#e05252')
axes[2].set_yticks(y_pos)
axes[2].set_yticklabels(hub_labels[::-1], fontsize=7.5, color='white')
axes[2].set_xlabel('SLA Breach Contribution (%)', color='white', fontsize=9)
axes[2].set_title('Operational Impact (SLA Breach %)', color='white', fontsize=11)
axes[2].tick_params(colors='white')

fig.suptitle(
    'Top 20 Bottleneck Hubs - Composite Ranking  (40% Betweenness + 40% SLA Breach + 20% Degree)',
    color='white', fontsize=13, y=1.02
)
plt.tight_layout()
plt.savefig(
    '../reports/visualizations/sla_breach_hubs.png',
    dpi=150, bbox_inches='tight', facecolor='#0f1117'
)
plt.close()
print('Hub ranking visualization saved.')

In [ ]:
# Visualization 4: Degree Distribution
in_degrees_list = [d for n, d in G.in_degree()]
out_degrees_list = [d for n, d in G.out_degree()]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.patch.set_facecolor('#0f1117')

for ax in axes:
    ax.set_facecolor('#1a1d27')
    for sp in ['top', 'right']:
        ax.spines[sp].set_visible(False)
    for sp in ['bottom', 'left']:
        ax.spines[sp].set_color('#555')

mean_in = round(np.mean(in_degrees_list), 1)
mean_out = round(np.mean(out_degrees_list), 1)

axes[0].hist(in_degrees_list, bins=30, color='#3a9bd5', alpha=0.85, edgecolor='#0f1117')
axes[0].axvline(x=np.mean(in_degrees_list), color='yellow', linestyle='--', linewidth=2,
                label='Mean: ' + str(mean_in))
axes[0].set_xlabel('In-Degree', color='white', fontsize=11)
axes[0].set_ylabel('Number of Hubs', color='white', fontsize=11)
axes[0].set_title('In-Degree Distribution (Routes Arriving at Hub)', color='white', fontsize=12)
axes[0].tick_params(colors='white')
axes[0].legend(facecolor='#1a1d27', edgecolor='#555', labelcolor='white')

axes[1].hist(out_degrees_list, bins=30, color='#e07b39', alpha=0.85, edgecolor='#0f1117')
axes[1].axvline(x=np.mean(out_degrees_list), color='yellow', linestyle='--', linewidth=2,
                label='Mean: ' + str(mean_out))
axes[1].set_xlabel('Out-Degree', color='white', fontsize=11)
axes[1].set_ylabel('Number of Hubs', color='white', fontsize=11)
axes[1].set_title('Out-Degree Distribution (Routes Departing from Hub)', color='white', fontsize=12)
axes[1].tick_params(colors='white')
axes[1].legend(facecolor='#1a1d27', edgecolor='#555', labelcolor='white')

fig.suptitle('Logistics Network Degree Distribution', color='white', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(
    '../reports/visualizations/degree_distribution.png',
    dpi=150, bbox_inches='tight', facecolor='#0f1117'
)
plt.close()
print('Degree distribution visualization saved.')

## 15. Task 2 Completion Summary

In [ ]:
print('=' * 60)
print('TASK 2 COMPLETE: BOTTLENECK & CORRIDOR AUDIT')
print('=' * 60)
print()
print('Reports saved to ../reports/:')
report_files = [
    'network_summary.csv',
    'degree_centrality_hubs.csv',
    'in_degree_hubs.csv',
    'out_degree_hubs.csv',
    'betweenness_centrality_hubs.csv',
    'clustering_coefficients.csv',
    'bottleneck_corridors.csv',
    'stratified_delay_analysis.csv',
    'sla_breach_corridors.csv',
    'sla_breach_hubs.csv',
    'hub_bottleneck_ranking.csv',
]
for f in report_files:
    path = '../reports/' + f
    status = 'SAVED' if os.path.exists(path) else 'MISSING'
    print('  [' + status + '] ' + f)
print()
print('Visualizations saved to ../reports/visualizations/:')
viz_files = [
    'bottleneck_network.png',
    'chronic_corridors.png',
    'sla_breach_hubs.png',
    'degree_distribution.png',
]
for v in viz_files:
    path = '../reports/visualizations/' + v
    status = 'SAVED' if os.path.exists(path) else 'MISSING'
    print('  [' + status + '] ' + v)